In [0]:
"""
Bean There - bronze data generator.
Single command:  python generate_bronze.py        (settings live in config.yaml)
"""
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

try:
    BASE_DIR = Path(__file__).resolve().parent   # terminal / python script
except NameError:
    BASE_DIR = Path.cwd()                        # Databricks editor / notebook

CONFIG_PATH = BASE_DIR / "config.yaml"

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# Use the configured random seed
SEED = config["seed"]
rng = np.random.default_rng(SEED)

# Read dataset settings
DATA_CONFIG = config["data"]
DEFECT_CONFIG = config["defects"]

print("Configuration loaded successfully!")
print("Random seed:", SEED)
print("Number of stores:", DATA_CONFIG["num_stores"])
print("Number of employees:", DATA_CONFIG["num_employees"])
print("Number of products:", DATA_CONFIG["num_products"])
print("Number of ingredients:", DATA_CONFIG["num_ingredients"])
print("Defects enabled:", DEFECT_CONFIG["enabled"])
print("Defect rate:", DEFECT_CONFIG["rate"])

# =============================================================================
# STEP 2: STORES
# =============================================================================
START_DATE = pd.Timestamp(DATA_CONFIG["start_date"])
END_DATE = pd.Timestamp(DATA_CONFIG["end_date"])


def generate_stores(n_stores, rng):
    # (city, state, region): 3 cities, 3 regions
    locations = [("Austin", "TX", "Central"),
                 ("Dallas", "TX", "North"),
                 ("Houston", "TX", "South")]
    zip_base = {"Austin": 78701, "Dallas": 75201, "Houston": 77002}
    streets = ["Main St", "Oak Ave", "Elm St", "Maple Dr", "Cedar Ln", "Market St"]

    rows = []
    for i in range(n_stores):
        city, state, region = locations[i % 3]          # spread stores across the 3 cities
        store_id = i + 1

        if i == n_stores - 1:
            # last store opens mid-period (required by the assignment)
            open_date = START_DATE + pd.Timedelta(days=180)
        else:
            # other stores opened before the data period starts
            open_date = pd.Timestamp("2019-01-01") + pd.Timedelta(days=int(rng.integers(0, 2000)))

        rows.append({
            "store_id": store_id,
            "store_name": f"Bean There {city} #{store_id}",
            "address": f"{100 + store_id * 37} {streets[i % len(streets)]}",
            "city": city,
            "state": state,
            "zip": str(zip_base[city] + store_id),
            "region": region,
            "open_date": open_date.strftime("%Y-%m-%d"),
            "square_footage": int(rng.integers(1400, 2500)),
            "has_drive_thru": (i % 2 == 0),             # mix of drive-thru and walk-in stores
        })
    return pd.DataFrame(rows)


stores = generate_stores(DATA_CONFIG["num_stores"], rng)
print("\n[stores] rows:", len(stores))

# =============================================================================
# STEP 3: EMPLOYEES
# =============================================================================
FIRST_NAMES = ["Olivia", "Liam", "Emma", "Noah", "Ava", "Elijah", "Sophia", "Mateo", "Isabella", "Lucas",
               "Mia", "Ethan", "Amelia", "Aiden", "Harper", "Jackson", "Priya", "Arjun", "Mei", "Omar"]
LAST_NAMES = ["Smith", "Johnson", "Garcia", "Martinez", "Lee", "Nguyen", "Patel", "Brown", "Davis",
              "Wilson", "Lopez", "Kim", "Clark", "Lewis", "Walker", "Young", "Khan", "Chen", "Torres", "Reed"]
BASE_RATE = {"barista": 15.0, "shift_lead": 18.5, "store_manager": 25.0}


def generate_employees(n_employees, stores, rng):
    n_stores = len(stores)
    per_store, extra = divmod(n_employees, n_stores)   # 60 employees / 12 stores = 5 each
    rows, emp_id = [], 1001

    # 1) Base roster: 1 manager, 1 shift lead, the rest baristas, per store
    for s in stores.itertuples():
        open_date = pd.Timestamp(s.open_date)
        n_here = per_store + (1 if s.store_id <= extra else 0)
        roles = ["store_manager", "shift_lead"] + ["barista"] * (n_here - 2)
        for role in roles:
            if open_date > START_DATE:
                # store opens mid-period: staff hired shortly before opening
                hire = open_date - pd.Timedelta(days=int(rng.integers(14, 45)))
            else:
                # never hired before the store existed
                earliest = max(open_date, START_DATE - pd.Timedelta(days=1800))
                span = max((START_DATE - earliest).days, 1)
                hire = earliest + pd.Timedelta(days=int(rng.integers(0, span)))
            rate = BASE_RATE[role] + 0.25 * int(rng.integers(0, 9))
            rows.append({"employee_id": emp_id, "first_name": str(rng.choice(FIRST_NAMES)),
                         "last_name": str(rng.choice(LAST_NAMES)), "hire_date": hire,
                         "termination_date": pd.NaT, "role": role, "home_store_id": s.store_id,
                         "hourly_rate": rate})
            emp_id += 1
    emp = pd.DataFrame(rows)
    start_rate = dict(zip(emp.employee_id, emp.hourly_rate))   # rate before any raise

    # 2) Terminate 6 baristas during the period and hire a replacement for each
    pool = emp[(emp.role == "barista") & (emp.hire_date < START_DATE)].index.to_numpy()
    leavers = rng.choice(pool, size=6, replace=False)
    new_rows = []
    for idx in leavers:
        term = START_DATE + pd.Timedelta(days=int(rng.integers(30, 300)))
        emp.at[idx, "termination_date"] = term
        new_rows.append({"employee_id": emp_id, "first_name": str(rng.choice(FIRST_NAMES)),
                         "last_name": str(rng.choice(LAST_NAMES)),
                         "hire_date": term - pd.Timedelta(days=int(rng.integers(0, 10))),
                         "termination_date": pd.NaT, "role": "barista",
                         "home_store_id": int(emp.at[idx, "home_store_id"]), "hourly_rate": 15.5})
        start_rate[emp_id] = 15.5
        emp_id += 1
    emp = pd.concat([emp, pd.DataFrame(new_rows)], ignore_index=True)

    # 3) Raises: 10 current employees get a raise on one of three dates
    active = emp[emp.termination_date.isna()].index.to_numpy()
    raise_idx = rng.choice(active, size=10, replace=False)
    history = []
    for idx in raise_idx:
        eff = pd.Timestamp(str(rng.choice(["2025-03-01", "2025-06-01", "2025-09-01"])))
        eid = int(emp.at[idx, "employee_id"])
        new_rate = round(start_rate[eid] + float(rng.choice([0.75, 1.00, 1.50])), 2)
        emp.at[idx, "hourly_rate"] = new_rate              # employees table holds the CURRENT rate
        history.append((eid, max(emp.at[idx, "hire_date"], START_DATE), eff - pd.Timedelta(days=1), start_rate[eid]))
        history.append((eid, eff, pd.NaT, new_rate))
    raised = {h[0] for h in history}
    for r in emp.itertuples():
        if r.employee_id not in raised:
            history.append((r.employee_id, max(r.hire_date, START_DATE), pd.NaT, start_rate[r.employee_id]))
    rate_history = pd.DataFrame(history, columns=["employee_id", "effective_date", "end_date", "hourly_rate"])
    return emp, rate_history


employees, employee_rate_history = generate_employees(DATA_CONFIG["num_employees"], stores, rng)
print("[employees] rows:", len(employees))

# =============================================================================
# STEP 4: INGREDIENTS, PRODUCTS, RECIPES
# =============================================================================
# name, type, unit_of_measure, supplier_id, starting unit_cost
INGREDIENT_LIST = [
    ("Whole Milk", "dairy", "ml", 1, 0.0035), ("Oat Milk", "dairy", "ml", 1, 0.0075),
    ("Almond Milk", "dairy", "ml", 1, 0.0070), ("Whipped Cream", "dairy", "ml", 1, 0.0090),
    ("Espresso Beans", "coffee_bean", "g", 2, 0.0250), ("House Blend Beans", "coffee_bean", "g", 2, 0.0180),
    ("Cold Brew Concentrate", "coffee_bean", "ml", 2, 0.0120), ("Loose Tea Leaves", "coffee_bean", "g", 3, 0.0300),
    ("Vanilla Syrup", "syrup", "ml", 3, 0.0120), ("Caramel Syrup", "syrup", "ml", 3, 0.0120),
    ("Pumpkin Spice Syrup", "syrup", "ml", 3, 0.0160), ("Peppermint Syrup", "syrup", "ml", 3, 0.0140),
    ("Mocha Sauce", "syrup", "ml", 3, 0.0150), ("Chai Concentrate", "syrup", "ml", 3, 0.0100),
    ("Matcha Powder", "syrup", "g", 3, 0.0800),
    ("Frozen Croissant", "bakery", "each", 4, 0.55), ("Muffin", "bakery", "each", 4, 0.50),
    ("Bagel", "bakery", "each", 4, 0.35), ("Cookie", "bakery", "each", 4, 0.40),
    ("Sandwich Bread", "bakery", "slice", 4, 0.15), ("Sandwich Filling", "bakery", "serving", 4, 1.10),
    ("Cup Small", "packaging", "each", 5, 0.12), ("Cup Medium", "packaging", "each", 5, 0.15),
    ("Cup Large", "packaging", "each", 5, 0.19), ("Lid", "packaging", "each", 5, 0.05),
]
# ingredient_name, effective_date, multiplier   (>= 3 cost changes required)
COST_CHANGES = [("Espresso Beans", "2025-05-01", 1.08), ("Oat Milk", "2025-03-01", 1.10),
                ("Whole Milk", "2025-08-01", 1.06), ("Frozen Croissant", "2025-06-15", 1.07)]

# name, category, subcategory, temperature, is_seasonal, launch, discontinued, starting price, popularity, recipe (MEDIUM size qty)
PRODUCT_LIST = [
    ("Latte", "espresso", "hot_espresso", "hot", False, "2019-01-01", None, 4.50, 10.0, [("Espresso Beans", 18), ("Whole Milk", 220)]),
    ("Iced Latte", "espresso", "iced_espresso", "iced", False, "2019-01-01", None, 4.75, 8.0, [("Espresso Beans", 18), ("Whole Milk", 160)]),
    ("Vanilla Latte", "espresso", "hot_espresso", "hot", False, "2019-01-01", None, 4.95, 6.0, [("Espresso Beans", 18), ("Whole Milk", 220), ("Vanilla Syrup", 20)]),
    ("Cappuccino", "espresso", "hot_espresso", "hot", False, "2019-01-01", None, 4.25, 5.0, [("Espresso Beans", 18), ("Whole Milk", 150)]),
    ("Americano", "espresso", "hot_espresso", "hot", False, "2019-01-01", None, 3.50, 5.0, [("Espresso Beans", 18)]),
    ("Iced Americano", "espresso", "iced_espresso", "iced", False, "2019-01-01", None, 3.75, 4.0, [("Espresso Beans", 18)]),
    ("Mocha", "espresso", "hot_espresso", "hot", False, "2019-01-01", None, 5.00, 4.0, [("Espresso Beans", 18), ("Whole Milk", 180), ("Mocha Sauce", 30), ("Whipped Cream", 15)]),
    ("Iced Mocha", "espresso", "iced_espresso", "iced", False, "2019-01-01", None, 5.25, 3.5, [("Espresso Beans", 18), ("Whole Milk", 140), ("Mocha Sauce", 30)]),
    ("Flat White", "espresso", "hot_espresso", "hot", False, "2020-03-01", None, 4.75, 3.0, [("Espresso Beans", 20), ("Whole Milk", 160)]),
    ("Caramel Macchiato", "espresso", "hot_espresso", "hot", False, "2019-01-01", None, 5.25, 4.0, [("Espresso Beans", 18), ("Whole Milk", 200), ("Vanilla Syrup", 10), ("Caramel Syrup", 15)]),
    ("Pumpkin Spice Latte", "espresso", "seasonal_espresso", "hot", True, "2025-09-01", "2025-11-30", 5.50, 9.0, [("Espresso Beans", 18), ("Whole Milk", 200), ("Pumpkin Spice Syrup", 25), ("Whipped Cream", 15)]),
    ("Peppermint Mocha", "espresso", "seasonal_espresso", "hot", True, "2025-11-15", "2025-12-28", 5.50, 8.0, [("Espresso Beans", 18), ("Whole Milk", 180), ("Mocha Sauce", 25), ("Peppermint Syrup", 15), ("Whipped Cream", 15)]),
    ("House Drip", "brewed", "drip", "hot", False, "2019-01-01", None, 2.75, 9.0, [("House Blend Beans", 16)]),
    ("Dark Roast Drip", "brewed", "drip", "hot", False, "2019-01-01", None, 2.95, 4.0, [("House Blend Beans", 16)]),
    ("Decaf Drip", "brewed", "drip", "hot", False, "2019-01-01", None, 2.75, 1.5, [("House Blend Beans", 16)]),
    ("Cold Brew", "brewed", "cold_brew", "iced", False, "2019-06-01", None, 4.25, 6.0, [("Cold Brew Concentrate", 120)]),
    ("Nitro Cold Brew", "brewed", "cold_brew", "iced", False, "2021-05-01", None, 5.25, 2.5, [("Cold Brew Concentrate", 140)]),
    ("Summer Caramel Cold Brew", "brewed", "seasonal_cold_brew", "iced", True, "2025-05-15", "2025-09-15", 5.25, 6.0, [("Cold Brew Concentrate", 120), ("Caramel Syrup", 20), ("Whole Milk", 60)]),
    ("Black Tea", "tea", "hot_tea", "hot", False, "2019-01-01", None, 3.00, 2.5, [("Loose Tea Leaves", 4)]),
    ("Iced Black Tea", "tea", "iced_tea", "iced", False, "2019-01-01", None, 3.25, 3.0, [("Loose Tea Leaves", 5)]),
    ("Green Tea", "tea", "hot_tea", "hot", False, "2019-01-01", None, 3.00, 2.0, [("Loose Tea Leaves", 4)]),
    ("Chai Latte", "tea", "latte_tea", "hot", False, "2019-01-01", None, 4.75, 4.0, [("Chai Concentrate", 120), ("Whole Milk", 150)]),
    ("Iced Chai", "tea", "iced_tea", "iced", False, "2019-01-01", None, 4.95, 3.0, [("Chai Concentrate", 100), ("Whole Milk", 130)]),
    ("Matcha Latte", "tea", "latte_tea", "hot", False, "2021-01-01", None, 5.25, 3.0, [("Matcha Powder", 5), ("Whole Milk", 220)]),
    ("Iced Matcha", "tea", "iced_tea", "iced", False, "2021-01-01", None, 5.50, 3.0, [("Matcha Powder", 5), ("Whole Milk", 170)]),
    ("Butter Croissant", "pastry", "croissant", "na", False, "2019-01-01", None, 3.25, 6.0, [("Frozen Croissant", 1)]),
    ("Almond Croissant", "pastry", "croissant", "na", False, "2020-01-01", None, 4.25, 2.5, [("Frozen Croissant", 1)]),
    ("Blueberry Muffin", "pastry", "muffin", "na", False, "2019-01-01", None, 3.25, 4.0, [("Muffin", 1)]),
    ("Plain Bagel", "pastry", "bagel", "na", False, "2019-01-01", None, 2.75, 3.0, [("Bagel", 1)]),
    ("Chocolate Chip Cookie", "pastry", "cookie", "na", False, "2019-01-01", None, 2.50, 3.5, [("Cookie", 1)]),
    ("Pumpkin Muffin", "pastry", "seasonal_muffin", "na", True, "2025-09-01", "2025-11-30", 3.75, 3.0, [("Muffin", 1)]),
    ("Egg & Cheese Sandwich", "sandwich", "breakfast", "na", False, "2019-01-01", None, 6.50, 3.0, [("Sandwich Bread", 2), ("Sandwich Filling", 1)]),
    ("Breakfast Sandwich", "sandwich", "breakfast", "na", False, "2020-01-01", None, 6.95, 2.5, [("Sandwich Bread", 2), ("Sandwich Filling", 2)]),
    ("Turkey Club", "sandwich", "lunch", "na", False, "2019-01-01", None, 8.50, 2.0, [("Sandwich Bread", 3), ("Sandwich Filling", 2)]),
    ("Ham & Swiss", "sandwich", "lunch", "na", False, "2019-01-01", None, 7.95, 1.8, [("Sandwich Bread", 2), ("Sandwich Filling", 1)]),
    ("Veggie Sandwich", "sandwich", "lunch", "na", False, "2020-01-01", None, 7.50, 1.5, [("Sandwich Bread", 2), ("Sandwich Filling", 1)]),
    ("Travel Tumbler", "merch", "drinkware", "na", False, "2019-01-01", None, 24.00, 0.4, []),
    ("Ceramic Mug", "merch", "drinkware", "na", False, "2019-01-01", None, 14.00, 0.4, []),
    ("Whole Bean Bag", "merch", "beans", "na", False, "2019-01-01", None, 16.00, 0.8, []),
    ("Canvas Tote", "merch", "bags", "na", False, "2019-01-01", None, 12.00, 0.3, []),
]
# product_name -> [(effective_date, new_base_price)]   (>= 3 price changes required)
PRICE_CHANGES = {"Latte": [("2025-04-01", 4.75)], "House Drip": [("2025-07-01", 2.95)],
                 "Butter Croissant": [("2025-03-15", 3.50)], "Cold Brew": [("2025-06-01", 4.50)]}

SIZES = ["small", "medium", "large"]
SIZE_QTY_FACTOR = {"small": 0.75, "medium": 1.0, "large": 1.30}   # recipe quantity per size
DRINK_CATEGORIES = {"espresso", "brewed", "tea"}                  # only drinks come in sizes


def generate_ingredients():
    ing = pd.DataFrame(INGREDIENT_LIST, columns=["ingredient_name", "ingredient_type", "unit_of_measure",
                                                 "supplier_id", "unit_cost"])
    ing.insert(0, "ingredient_id", range(1, len(ing) + 1))
    history, current = [], {}
    for r in ing.itertuples():
        cost, start = r.unit_cost, START_DATE
        for name, eff, mult in sorted([c for c in COST_CHANGES if c[0] == r.ingredient_name], key=lambda c: c[1]):
            history.append((r.ingredient_id, start, pd.Timestamp(eff) - pd.Timedelta(days=1), cost))
            start, cost = pd.Timestamp(eff), round(cost * mult, 5)
        history.append((r.ingredient_id, start, pd.NaT, cost))
        current[r.ingredient_id] = cost
    ing["unit_cost"] = ing.ingredient_id.map(current)         # export holds the CURRENT cost
    cost_history = pd.DataFrame(history, columns=["ingredient_id", "effective_date", "end_date", "unit_cost"])
    return ing, cost_history


def generate_products_and_recipes(ingredients):
    ing_id = dict(zip(ingredients.ingredient_name, ingredients.ingredient_id))
    ing_unit = dict(zip(ingredients.ingredient_name, ingredients.unit_of_measure))
    prods, attrs, recs, history = [], [], [], []
    for pid, (name, cat, sub, temp, seasonal, launch, disc, price, pop, recipe) in enumerate(PRODUCT_LIST, start=1):
        changes = sorted(PRICE_CHANGES.get(name, []))
        prods.append({"product_id": pid, "product_name": name, "category": cat, "subcategory": sub,
                      "is_seasonal": seasonal, "launch_date": launch, "discontinued_date": disc,
                      "base_price": changes[-1][1] if changes else price})   # current list price
        attrs.append({"product_id": pid, "temperature": temp, "popularity": pop})
        cur, start = price, max(pd.Timestamp(launch), START_DATE)
        for eff, new_price in changes:
            history.append((pid, start, pd.Timestamp(eff) - pd.Timedelta(days=1), cur))
            start, cur = pd.Timestamp(eff), new_price
        history.append((pid, start, pd.Timestamp(disc) if disc else pd.NaT, cur))
        for size in (SIZES if cat in DRINK_CATEGORIES else [None]):      # food: one recipe, size = NULL
            factor = SIZE_QTY_FACTOR[size] if size else 1.0
            for ing_name, qty in recipe:
                recs.append((pid, size, ing_id[ing_name], round(qty * factor, 2), ing_unit[ing_name]))
            if size:                                                      # every drink needs a cup and a lid
                recs.append((pid, size, ing_id[f"Cup {size.title()}"], 1, "each"))
                recs.append((pid, size, ing_id["Lid"], 1, "each"))
    products = pd.DataFrame(prods)
    product_attrs = pd.DataFrame(attrs)
    recipes = pd.DataFrame(recs, columns=["product_id", "size", "ingredient_id", "quantity_used", "unit_of_measure"])
    price_history = pd.DataFrame(history, columns=["product_id", "effective_date", "end_date", "base_price"])
    return products, product_attrs, recipes, price_history


ingredients, ingredient_cost_history = generate_ingredients()
products, product_attrs, recipes, product_price_history = generate_products_and_recipes(ingredients)

assert len(ingredients) == DATA_CONFIG["num_ingredients"], "ingredient count does not match config.yaml"
assert len(products) == DATA_CONFIG["num_products"], "product count does not match config.yaml"
print("[ingredients] rows:", len(ingredients), "| [products] rows:", len(products), "| [recipes] rows:", len(recipes))

# =============================================================================
# STEP 5: LOYALTY MEMBERS + TIER HISTORY
# =============================================================================
TIER_WEIGHT = {"bronze": 1.0, "silver": 2.0, "gold": 4.0}   # higher tier -> visits more often


def generate_loyalty_members(n_members, stores, rng):
    n = n_members
    ids = np.arange(50001, 50001 + n)
    period_days = (END_DATE - START_DATE).days

    # ~30% of members sign up DURING the period, the rest before it
    in_period = rng.random(n) < 0.30
    pre_start = pd.Timestamp("2022-01-01")
    in_dates = START_DATE + pd.to_timedelta(rng.integers(0, period_days - 20, n), unit="D")
    pre_dates = pre_start + pd.to_timedelta(rng.integers(0, (START_DATE - pre_start).days, n), unit="D")
    signup = pd.to_datetime(np.where(in_period, in_dates, pre_dates))

    # signup store must already be open on the signup date
    store_open = pd.to_datetime(stores.open_date).values
    store_ids = stores.store_id.values
    signup_store = [int(rng.choice(store_ids[store_open <= np.datetime64(sd)])) for sd in signup]

    first = rng.choice(FIRST_NAMES, n)
    last = rng.choice(LAST_NAMES, n)
    members = pd.DataFrame({
        "customer_id": ids,
        "first_name": first,
        "last_name": last,
        "email": [f"{f.lower()}.{l.lower()}{i}@example.com" for f, l, i in zip(first, last, ids)],
        "birth_date": pd.Timestamp("1960-01-01") + pd.to_timedelta(rng.integers(0, 17000, n), unit="D"),
        "signup_date": signup,
        "signup_store_id": signup_store,
    })

    # Tier history: everyone starts bronze; some upgrade to silver, some of those to gold,
    # and a few gold members drop back to silver.
    hist, final_tier = [], {}
    for cid, sd in zip(ids, signup):
        tiers = [("bronze", sd)]
        if rng.random() < 0.30:
            lo, hi = sd + pd.Timedelta(days=45), pd.Timestamp("2025-10-15")
            if lo < hi:
                silver = lo + pd.Timedelta(days=int(rng.integers(0, (hi - lo).days)))
                tiers.append(("silver", silver))
                if rng.random() < 0.35:
                    lo2, hi2 = silver + pd.Timedelta(days=45), pd.Timestamp("2025-12-01")
                    if lo2 < hi2:
                        gold = lo2 + pd.Timedelta(days=int(rng.integers(0, (hi2 - lo2).days)))
                        tiers.append(("gold", gold))
                        if rng.random() < 0.12 and gold + pd.Timedelta(days=70) < END_DATE:
                            tiers.append(("silver", gold + pd.Timedelta(days=int(rng.integers(60, 70)))))
        for j, (tier, eff) in enumerate(tiers):
            end = tiers[j + 1][1] - pd.Timedelta(days=1) if j + 1 < len(tiers) else pd.NaT
            hist.append((cid, tier, eff, end))
        final_tier[cid] = tiers[-1][0]

    members["current_tier"] = members.customer_id.map(final_tier)
    members["visit_weight"] = members.current_tier.map(TIER_WEIGHT)   # helper, not exported
    tier_history = pd.DataFrame(hist, columns=["customer_id", "tier", "effective_date", "end_date"])
    return members, tier_history


loyalty_members, loyalty_tier_history = generate_loyalty_members(DATA_CONFIG["num_loyalty_members"], stores, rng)
print("[loyalty_members] rows:", len(loyalty_members), "| [loyalty_tier_history] rows:", len(loyalty_tier_history))

# =============================================================================
# STEP 6a: ORDER SKELETON
# =============================================================================
WARM = {1: 0.0, 2: 0.05, 3: 0.25, 4: 0.45, 5: 0.70, 6: 0.95, 7: 1.0, 8: 1.0, 9: 0.75, 10: 0.40, 11: 0.15, 12: 0.05}

HOURS = np.arange(6, 20)       # stores take orders from 06:00 to 19:59
HOUR_PROFILE = {
    "weekday_commuter": [0.05, 0.16, 0.18, 0.12, 0.06, 0.05, 0.07, 0.06, 0.07, 0.05, 0.03, 0.02, 0.015, 0.01],
    "weekday_walkin":   [0.03, 0.10, 0.13, 0.11, 0.08, 0.07, 0.08, 0.07, 0.07, 0.06, 0.05, 0.04, 0.03, 0.02],
    "weekend":          [0.00, 0.05, 0.09, 0.12, 0.12, 0.10, 0.08, 0.07, 0.07, 0.06, 0.05, 0.04, 0.03, 0.02],
}
CHANNELS = ["in_store", "mobile_app", "drive_thru"]


def generate_order_skeleton(stores, rng, avg_orders):
    days = pd.date_range(START_DATE, END_DATE, freq="D")
    # each store gets its own volume level (some busy, some slow)
    store_mult = {sid: float(rng.uniform(0.80, 1.40)) for sid in stores.store_id}
    # each store also has its own share of loyalty customers (customer base differs by store)
    loyalty_level = {sid: float(rng.uniform(0.70, 1.30)) for sid in stores.store_id}
    parts = []

    for s in stores.itertuples():
        open_date = pd.Timestamp(s.open_date)
        for day in days:
            if day < open_date:                       # a store sells nothing before it opens
                continue
            weekend = day.dayofweek >= 5
            # drive-thru stores: busier on weekdays (commuters). Walk-in stores: busier on weekends.
            if s.has_drive_thru:
                dow_mult = 0.82 if weekend else 1.10
            else:
                dow_mult = 1.18 if weekend else 0.94
            season_mult = 1.0 + 0.08 * (1 - WARM[day.month])             # slightly busier in cold months
            growth = 1.0 + 0.012 * (day.month - 1 + day.day / 30)         # slow upward trend over the year
            lam = avg_orders * store_mult[s.store_id] * dow_mult * season_mult * growth * rng.lognormal(0, 0.10)
            n = int(rng.poisson(lam))
            if n == 0:
                continue

            if weekend:
                profile = HOUR_PROFILE["weekend"]
            elif s.has_drive_thru:
                profile = HOUR_PROFILE["weekday_commuter"]
            else:
                profile = HOUR_PROFILE["weekday_walkin"]
            profile = np.array(profile) / np.sum(profile)
            hours = rng.choice(HOURS, n, p=profile)

            mobile_share = 0.12 + 0.012 * (day.month - 1)                 # mobile ordering grows through the year
            dt_share = (0.28 if weekend else 0.38) if s.has_drive_thru else 0.0
            channel = rng.choice(CHANNELS, n, p=[1 - mobile_share - dt_share, mobile_share, dt_share])
            p_loyalty = np.where(channel == "mobile_app", 0.68, np.where(channel == "drive_thru", 0.32, 0.38))
            p_loyalty = np.clip(p_loyalty * loyalty_level[s.store_id], 0, 0.95)

            parts.append(pd.DataFrame({
                "store_id": s.store_id,
                "order_date": day,
                "hour": hours,
                "minute": rng.integers(0, 60, n),
                "second": rng.integers(0, 60, n),
                "order_channel": channel,
                "is_loyalty": rng.random(n) < p_loyalty,                  # linked to a member in Step 6c
            }))

    o = pd.concat(parts, ignore_index=True)
    o["order_timestamp"] = (o.order_date + pd.to_timedelta(o.hour, unit="h")
                            + pd.to_timedelta(o.minute, unit="m") + pd.to_timedelta(o.second, unit="s"))
    o = o.sort_values(["order_timestamp", "store_id"], kind="stable").reset_index(drop=True)
    o.insert(0, "order_id", 100001 + np.arange(len(o)))
    return o


orders = generate_order_skeleton(stores, rng, DATA_CONFIG["avg_orders_per_store_day"])
print("[orders skeleton] rows:", len(orders))

# =============================================================================
# STEP 6b: ORDER ITEMS
# =============================================================================
SIZE_PRICE_ADJ = {"small": -0.50, "medium": 0.0, "large": 0.60}   # price vs the list (medium) price
MODIFIERS = np.array(["oat milk", "almond milk", "extra shot", "vanilla", "caramel",
                      "no whip", "decaf", "extra hot", "light ice"])


def generate_order_items(orders, products, product_attrs, product_price_history, stores, rng):
    days = pd.date_range(START_DATE, END_DATE, freq="D")
    P = len(products)
    attrs = product_attrs.set_index("product_id").reindex(products.product_id)
    pop = attrs.popularity.to_numpy(dtype=float)
    temp = attrs.temperature.to_numpy()
    cat = products.category.to_numpy()
    is_drink = np.isin(cat, list(DRINK_CATEGORIES))
    launch = pd.to_datetime(products.launch_date).to_numpy()
    discont = pd.to_datetime(products.discontinued_date.fillna("2100-01-01")).to_numpy()

    # price in effect for every product on every day (this is how unit_price follows price changes)
    price_by_day = np.zeros((len(days), P))
    for pid in range(P):
        h = product_price_history[product_price_history.product_id == pid + 1]
        for r in h.itertuples():
            end = r.end_date if pd.notna(r.end_date) else END_DATE
            price_by_day[(days >= r.effective_date) & (days <= end), pid] = r.base_price

    # each store has its own product mix (store variation)
    store_pref = {}
    for sid in stores.store_id:
        pref = np.ones(P)
        pref[cat == "pastry"] *= rng.uniform(0.7, 1.5)
        pref[cat == "sandwich"] *= rng.uniform(0.6, 1.6)
        pref[cat == "brewed"] *= rng.uniform(0.7, 1.4)
        pref[cat == "merch"] *= rng.uniform(0.5, 2.0)
        pref[temp == "iced"] *= rng.uniform(0.8, 1.3)
        store_pref[sid] = pref

    ids, pidx_all, size_all, qty_all, price_all = [], [], [], [], []
    is_loy = orders.is_loyalty.to_numpy()
    order_ids = orders.order_id.to_numpy()

    for (sid, day), idx in orders.groupby(["store_id", "order_date"]).indices.items():
        di = (day - START_DATE).days
        n_items = np.clip(1 + rng.poisson(1.20 + 0.25 * is_loy[idx]), 1, 7)   # loyalty members buy bigger baskets
        total = int(n_items.sum())

        available = ((launch <= np.datetime64(day)) & (discont >= np.datetime64(day))).astype(float)
        season = np.where(temp == "iced", 0.45 + 1.4 * WARM[day.month],        # iced drinks sell more when warm
                          np.where(temp == "hot", 1.55 - 1.0 * WARM[day.month], 1.0))
        w = available * pop * season * store_pref[sid]
        pidx = rng.choice(P, total, p=w / w.sum())

        size_code = np.where(is_drink[pidx], rng.choice(3, total, p=[0.25, 0.50, 0.25]), -1)
        size = np.array(["small", "medium", "large", None], dtype=object)[size_code]   # -1 -> None (food, merch)
        adj = np.select([size_code == 0, size_code == 2], [SIZE_PRICE_ADJ["small"], SIZE_PRICE_ADJ["large"]], 0.0)

        ids.append(np.repeat(order_ids[idx], n_items))
        pidx_all.append(pidx)
        size_all.append(size)
        qty_all.append(rng.choice([1, 2, 3], total, p=[0.90, 0.08, 0.02]))
        price_all.append(np.round(price_by_day[di, pidx] + adj, 2))

    items = pd.DataFrame({
        "order_id": np.concatenate(ids),
        "product_id": np.concatenate(pidx_all) + 1,
        "size": np.concatenate(size_all),
        "quantity": np.concatenate(qty_all),
        "unit_price": np.concatenate(price_all),
    })
    items = items.sort_values("order_id", kind="stable").reset_index(drop=True)
    items.insert(0, "order_item_id", 700001 + np.arange(len(items)))
    items["item_discount"] = 0.0          # filled in Step 6c
    items["promo_code"] = None            # filled in Step 6c

    # modifiers on ~35% of drinks
    n = len(items)
    drink_line = items["size"].notna().to_numpy()
    has_mod = drink_line & (rng.random(n) < 0.35)
    m1, m2 = rng.choice(MODIFIERS, n), rng.choice(MODIFIERS, n)
    two = has_mod & (rng.random(n) < 0.3) & (m1 != m2)
    text = np.where(two, np.char.add(np.char.add(m1.astype(str), ", "), m2.astype(str)), m1)
    items["modifiers"] = np.where(has_mod, text, None)
    return items


order_items = generate_order_items(orders, products, product_attrs, product_price_history, stores, rng)
print("[order_items] rows:", len(order_items))

# =============================================================================
# STEP 6c: CUSTOMERS, PROMOS, PAYMENT, TOTALS
# =============================================================================
# promo_code, description, type, value, start, end
PROMOS = [
    ("SUMMERSIP", "20% off iced drinks", "percent", 0.20, "2025-06-01", "2025-08-31"),
    ("FALLFEST", "15% off espresso drinks and pastries", "percent", 0.15, "2025-09-15", "2025-11-30"),
    ("HOLIDAY10", "10% off everything except merch", "percent", 0.10, "2025-12-01", "2025-12-31"),
    ("SPRINGPASTRY", "25% off pastries", "percent", 0.25, "2025-03-01", "2025-04-15"),
    ("WELCOME15", "15% off first order within 14 days of loyalty signup", "percent", 0.15, "2025-01-01", "2025-12-31"),
    ("LOYALTY_REWARD", "$3 off when redeeming 100 points", "fixed", 3.00, "2025-01-01", "2025-12-31"),
]
promo_codes = pd.DataFrame(PROMOS, columns=["promo_code", "description", "discount_type",
                                            "discount_value", "start_date", "end_date"])
TAX_RATE = 0.0825


def link_customers(orders, members, rng):
    """Pick a member for every order flagged is_loyalty (only members who have already signed up)."""
    m_ids = members.customer_id.to_numpy()
    m_signup = members.signup_date.to_numpy()
    m_store = members.signup_store_id.to_numpy()
    base_w = members.visit_weight.to_numpy(dtype=float)            # gold visits more than bronze
    store_arr = orders.store_id.to_numpy()
    flag = orders.is_loyalty.to_numpy()
    customer = np.full(len(orders), -1, dtype=np.int64)

    for day, idx in orders.groupby("order_date").indices.items():
        loy = idx[flag[idx]]
        if len(loy) == 0:
            continue
        d64 = np.datetime64(day)
        active = m_signup <= d64
        recent = (d64 - m_signup) <= np.timedelta64(14, "D")
        w = base_w * active * np.where(recent, 4.0, 1.0)            # new members come in soon after signing up
        for sid in np.unique(store_arr[loy]):
            sub = loy[store_arr[loy] == sid]
            ws = w * np.where(m_store == sid, 4.0, 1.0)             # members favour their signup store
            customer[sub] = rng.choice(m_ids, len(sub), p=ws / ws.sum())
    orders["customer_id"] = pd.Series(customer, index=orders.index).where(lambda s: s > 0).astype("Int64")
    return orders


def apply_promos(orders, items, products, product_attrs, members, rng):
    items = items.copy()
    cat = items.product_id.map(dict(zip(products.product_id, products.category)))
    temp = items.product_id.map(dict(zip(product_attrs.product_id, product_attrs.temperature)))
    gross = items.unit_price * items.quantity

    # 1) seasonal promos: ~20% of orders inside a promo window use that promo
    chosen = pd.Series(None, index=orders.order_id, dtype=object)
    for code, _, _, val, start, end in PROMOS[:4]:
        in_window = ((orders.order_date >= start) & (orders.order_date <= end)).to_numpy()
        free = chosen.reset_index(drop=True).isna().to_numpy()
        pick = in_window & free & (rng.random(len(orders)) < 0.20)
        chosen.loc[orders.order_id.to_numpy()[pick]] = code
    item_code = items.order_id.map(chosen)
    rules = {"SUMMERSIP": temp == "iced",
             "FALLFEST": cat.isin(["espresso", "pastry"]),
             "HOLIDAY10": cat != "merch",
             "SPRINGPASTRY": cat == "pastry"}
    for code, _, _, val, _, _ in PROMOS[:4]:
        mask = (item_code == code) & rules[code]
        items.loc[mask, "item_discount"] = (val * gross[mask]).round(2)
        items.loc[mask, "promo_code"] = code

    # 2) WELCOME15: a member's first orders within 14 days of signup (if no other promo already applied)
    promo_orders = set(items.loc[items.promo_code.notna(), "order_id"])
    signup = orders.customer_id.map(dict(zip(members.customer_id, members.signup_date)))
    days_since = (orders.order_date - pd.to_datetime(signup)).dt.days
    eligible = (orders.customer_id.notna() & (days_since <= 14) & (~orders.order_id.isin(promo_orders))
                & (rng.random(len(orders)) < 0.5)).to_numpy()
    welcome_orders = set(orders.order_id[eligible])
    mask = items.order_id.isin(welcome_orders)
    items.loc[mask, "item_discount"] = (0.15 * gross[mask]).round(2)
    items.loc[mask, "promo_code"] = "WELCOME15"
    return items


def assign_payment_and_tip_rules(orders, rng):
    n = len(orders)
    mobile = (orders.order_channel == "mobile_app").to_numpy()
    orders["payment_method"] = np.where(mobile, rng.choice(["credit", "app_wallet"], n, p=[0.25, 0.75]),
                                        rng.choice(["cash", "credit", "debit"], n, p=[0.18, 0.55, 0.27]))
    orders["tip_rate"] = rng.choice([0.0, 0.10, 0.15, 0.20], n, p=[0.35, 0.25, 0.25, 0.15])
    orders["tip_flat"] = rng.choice([0.0, 0.5, 1.0], n, p=[0.7, 0.2, 0.1])      # cash tips are small flat amounts
    return orders


def compute_order_totals(orders, items):
    """Order totals are DERIVED from the line items. Call again whenever item discounts change."""
    o = orders.drop(columns=[c for c in ["subtotal", "discount_total", "tax", "tip"] if c in orders.columns])
    line = (items.unit_price * items.quantity).round(2)
    agg = pd.DataFrame({"order_id": items.order_id, "gross": line, "disc": items.item_discount}) \
            .groupby("order_id").sum()
    o = o.merge(agg, left_on="order_id", right_index=True, how="left")
    o["subtotal"] = o.gross.round(2)
    o["discount_total"] = o.disc.round(2)
    o["tax"] = ((o.subtotal - o.discount_total) * TAX_RATE).round(2)
    tip = np.where(o.payment_method == "cash", o.tip_flat, (o.subtotal * o.tip_rate).round(2))
    o["tip"] = np.minimum(tip, o.subtotal).round(2)
    return o.drop(columns=["gross", "disc"])


orders = link_customers(orders, loyalty_members, rng)
order_items = apply_promos(orders, order_items, products, product_attrs, loyalty_members, rng)
orders = assign_payment_and_tip_rules(orders, rng)
orders = compute_order_totals(orders, order_items)
print("[orders] customers linked, promos applied, totals computed")

# =============================================================================
# STEP 7: LOYALTY TRANSACTIONS
# =============================================================================
TIER_MULT = {"bronze": 1.0, "silver": 1.25, "gold": 1.5}
REDEEM_POINTS = 100
REWARD_VALUE = 3.00
TIER_RANK = {"bronze": 0, "silver": 1, "gold": 2}


def generate_loyalty_transactions(orders, order_items, members, tier_history, rng):
    order_items = order_items.copy()

    # 1) member's tier on each order date (as-of join on the tier history)
    lo = orders.loc[orders.customer_id.notna(), ["order_id", "customer_id", "order_date"]].copy()
    lo["customer_id"] = lo.customer_id.astype("int64")
    lo = lo.sort_values("order_date")
    th = tier_history.sort_values("effective_date")[["customer_id", "tier", "effective_date"]]
    lo = pd.merge_asof(lo, th, left_on="order_date", right_on="effective_date",
                       by="customer_id", direction="backward")
    tier_on_order = dict(zip(lo.order_id, lo.tier))

    # 2) walk the loyalty orders in time order, tracking each member's points balance
    gross = (order_items.unit_price * order_items.quantity).to_numpy()
    disc = order_items.item_discount.to_numpy().copy()
    promo = order_items.promo_code.to_numpy(dtype=object).copy()
    rows_by_order = order_items.groupby("order_id").indices          # order_id -> row positions
    ordered = orders[orders.customer_id.notna()].sort_values("order_timestamp")

    balance, txns = {}, []
    for r in ordered.itertuples(index=False):
        cid = int(r.customer_id)
        pos = rows_by_order[r.order_id]
        has_promo = pd.notna(promo[pos]).any()
        bal = balance.get(cid, 0)
        if bal >= REDEEM_POINTS and not has_promo and rng.random() < 0.18:
            first = pos[0]                                            # reward comes off the first item
            disc[first] = min(REWARD_VALUE, gross[first])
            promo[first] = "LOYALTY_REWARD"
            bal -= REDEEM_POINTS
            txns.append((cid, r.order_id, "redeem", -REDEEM_POINTS, r.order_date))
        net = gross[pos].sum() - disc[pos].sum()                      # points are earned on the net amount
        pts = int(np.floor(net * TIER_MULT[tier_on_order[r.order_id]]))
        balance[cid] = bal + pts
        txns.append((cid, r.order_id, "earn", pts, r.order_date))

    order_items["item_discount"] = np.round(disc, 2)
    order_items["promo_code"] = promo

    # 3) bonuses: signup bonus (members who joined during the period) and tier-upgrade bonus
    for m in members[members.signup_date >= START_DATE].itertuples():
        txns.append((m.customer_id, None, "bonus", 50, m.signup_date))
    h = tier_history.sort_values(["customer_id", "effective_date"]).copy()
    h["rank"] = h.tier.map(TIER_RANK)
    h["prev_rank"] = h.groupby("customer_id")["rank"].shift(1)
    upgrades = h[h.prev_rank.notna() & (h["rank"] > h.prev_rank) & (h.effective_date >= START_DATE)]
    for u in upgrades.itertuples():
        txns.append((u.customer_id, None, "bonus", 100, u.effective_date))

    lt = pd.DataFrame(txns, columns=["customer_id", "order_id", "txn_type", "points", "txn_date"])

    # 4) expirations: ~8% of members lose 20% of their running balance at a month end
    expire = []
    for cid in rng.choice(members.customer_id.to_numpy(), int(0.08 * len(members)), replace=False):
        month_end = pd.Timestamp(2025, int(rng.integers(6, 13)), 1) + pd.offsets.MonthEnd(0)
        mine = lt[(lt.customer_id == cid) & (lt.txn_date <= month_end)]
        running = int(mine.points.sum())
        if running >= 100:
            expire.append((cid, None, "expire", -int(running * 0.2), month_end))
    if expire:
        lt = pd.concat([lt, pd.DataFrame(expire, columns=lt.columns)], ignore_index=True)

    type_order = {"bonus": 0, "earn": 1, "redeem": 2, "expire": 3}
    lt["_o"] = lt.txn_type.map(type_order)
    lt = lt.sort_values(["txn_date", "_o", "customer_id"]).drop(columns="_o").reset_index(drop=True)

    # an expiry must never push a member's balance below zero: drop it if it would
    lt["_bal"] = lt.groupby("customer_id").points.cumsum()
    bad_members = lt.loc[lt._bal < 0, "customer_id"].unique()
    lt = lt[~((lt.txn_type == "expire") & lt.customer_id.isin(bad_members))].drop(columns="_bal").reset_index(drop=True)

    lt.insert(0, "loyalty_txn_id", 900001 + np.arange(len(lt)))
    lt["order_id"] = lt.order_id.astype("Int64")
    return lt, order_items


loyalty_transactions, order_items = generate_loyalty_transactions(
    orders, order_items, loyalty_members, loyalty_tier_history, rng)
orders = compute_order_totals(orders, order_items)       # reward discounts changed the item discounts
print("[loyalty_transactions] rows:", len(loyalty_transactions))

# =============================================================================
# STEP 8: SHIFTS  (who worked where, and when)
# =============================================================================
def generate_shifts(orders, employees, employee_rate_history, stores, rng):
    # how many orders each store had on each day -> busier days get more staff
    orders_per_day = orders.groupby(["store_id", "order_date"]).size()

    # each store is a bit over- or under-staffed (1.0 = normal)
    staffing_level = {sid: float(rng.choice([0.7, 0.85, 1.0, 1.0, 1.15, 1.35])) for sid in stores.store_id}

    shift_types = ["open", "close", "mid"]
    rows = []
    for (store_id, day), n_orders in orders_per_day.items():
        # people who work at this store AND are employed on this day
        team = employees[employees.home_store_id == store_id]
        employed = team[(team.hire_date <= day) & (team.termination_date.isna() | (team.termination_date >= day))]
        if len(employed) == 0:
            continue

        # about 1 person for every 6 orders, at least 2, never more than the team size
        n_staff = int(round(n_orders / 6 * staffing_level[store_id]))
        n_staff = max(2, min(n_staff, len(employed)))
        chosen = rng.choice(employed.employee_id.to_numpy(), n_staff, replace=False)

        is_weekend = day.dayofweek >= 5
        for k, emp_id in enumerate(chosen):
            kind = shift_types[k % 3]                       # open, close, mid, open, close, ...
            if kind == "open":
                start_hour, end_hour = (6.5 if is_weekend else 5.5), 14.0
            elif kind == "close":
                start_hour, end_hour = 12.5, 20.5
            else:
                start_hour, end_hour = 10.0, 16.0
            rows.append((int(emp_id), int(store_id),
                         day + pd.Timedelta(hours=start_hour), day + pd.Timedelta(hours=end_hour)))

    shifts = pd.DataFrame(rows, columns=["employee_id", "store_id", "shift_start", "shift_end"])
    shifts = shifts.sort_values(["shift_start", "store_id", "employee_id"]).reset_index(drop=True)

    # pay rate on the day of the shift (this is where the raises show up)
    history = employee_rate_history.sort_values("effective_date")[["employee_id", "effective_date", "hourly_rate"]]
    shifts = pd.merge_asof(shifts, history, left_on="shift_start", right_on="effective_date",
                           by="employee_id", direction="backward")
    shifts = shifts.rename(columns={"hourly_rate": "hourly_rate_at_shift"}).drop(columns="effective_date")
    shifts.insert(0, "shift_id", 1 + np.arange(len(shifts)))
    return shifts


def assign_cashiers(orders, shifts, rng):
    """Every order is rung up by someone who is on shift at that store at that time."""
    cashier = np.zeros(len(orders), dtype=np.int64)
    shift_rows_by_day = shifts.groupby(["store_id", shifts.shift_start.dt.normalize()]).indices
    shift_emp = shifts.employee_id.to_numpy()
    shift_start = shifts.shift_start.to_numpy()
    shift_end = shifts.shift_end.to_numpy()
    order_time = orders.order_timestamp.to_numpy()

    for (store_id, day), order_rows in orders.groupby(["store_id", "order_date"]).indices.items():
        shift_rows = shift_rows_by_day[(store_id, day)]
        for i in order_rows:
            on_shift = shift_rows[(shift_start[shift_rows] <= order_time[i]) & (shift_end[shift_rows] > order_time[i])]
            if len(on_shift) == 0:                # nobody exactly on shift: use anyone working that day
                on_shift = shift_rows
            cashier[i] = shift_emp[rng.choice(on_shift)]
    orders["employee_id"] = cashier
    return orders


shifts = generate_shifts(orders, employees, employee_rate_history, stores, rng)
orders = assign_cashiers(orders, shifts, rng)
print("[shifts] rows:", len(shifts))

# =============================================================================
# STEP 9: INVENTORY SNAPSHOTS + WASTE LOG
# =============================================================================
def calculate_daily_usage(orders, order_items, recipes):
    """Ingredients used per store per day = (items sold) x (recipe quantity)."""
    items = order_items.merge(orders[["order_id", "store_id", "order_date"]], on="order_id")
    items["size"] = items["size"].fillna("none")           # food has no size
    rec = recipes.copy()
    rec["size"] = rec["size"].fillna("none")
    usage = items.merge(rec[["product_id", "size", "ingredient_id", "quantity_used"]], on=["product_id", "size"])
    usage["used"] = usage.quantity * usage.quantity_used
    return usage.groupby(["store_id", "order_date", "ingredient_id"]).used.sum().reset_index()


WASTE_CHANCE = {"dairy": 0.18, "bakery": 0.18, "syrup": 0.035, "coffee_bean": 0.045, "packaging": 0.02}
WASTE_REASONS = {"dairy": ["expired", "expired", "spilled", "remade_drink", "damaged"],
                 "bakery": ["expired", "expired", "expired", "damaged", "spilled"],
                 "syrup": ["spilled", "remade_drink", "expired", "damaged"],
                 "coffee_bean": ["spilled", "expired", "remade_drink", "damaged"],
                 "packaging": ["damaged", "damaged", "damaged", "spilled"]}


def generate_inventory_and_waste(usage, ingredients, stores, rng):
    all_days = pd.date_range(START_DATE, END_DATE, freq="D")
    ingredient_ids = ingredients.ingredient_id.to_numpy()
    ingredient_types = ingredients.ingredient_type.to_numpy()
    n_ing = len(ingredient_ids)

    snapshot_parts, waste_rows = [], []
    for s in stores.itertuples():
        days = all_days[all_days >= pd.Timestamp(s.open_date)]        # nothing before the store opens
        n_days = len(days)

        # table of usage: one row per day, one column per ingredient (0 if not used)
        used = (usage[usage.store_id == s.store_id]
                .pivot_table(index="order_date", columns="ingredient_id", values="used", aggfunc="sum")
                .reindex(index=days, columns=ingredient_ids).fillna(0).to_numpy())
        avg_use = np.maximum(used.mean(axis=0), 0.1)                  # average daily use per ingredient

        # --- waste: some stores waste more than others
        store_waste_level = float(rng.choice([0.6, 1.0, 1.0, 1.4, 2.0]))
        wasted = np.zeros_like(used)
        for j in range(n_ing):
            kind = ingredient_types[j]
            happens = (rng.random(n_days) < WASTE_CHANCE[kind] * store_waste_level) & (used[:, j] > 0)
            amount = np.round(np.maximum(avg_use[j] * rng.uniform(0.03, 0.25, n_days), 0.01), 2)
            for k in np.where(happens)[0]:
                wasted[k, j] = amount[k]
                waste_rows.append((days[k], s.store_id, int(ingredient_ids[j]), amount[k],
                                   str(rng.choice(WASTE_REASONS[kind]))))

        # --- inventory: stock goes down with use + waste, and is refilled on delivery days (Mon, Thu)
        days_of_cover = float(rng.choice([2.8, 4.5, 4.5, 5.5, 5.5]))  # low cover = stock runs low before delivery
        par_level = avg_use * days_of_cover * 1.8
        reorder_threshold = avg_use * 1.6
        on_hand = par_level.copy()
        on_hand_by_day = np.zeros_like(used)
        for k, day in enumerate(days):
            if day.dayofweek in (0, 3):
                on_hand = np.maximum(on_hand, par_level * rng.uniform(0.95, 1.05, n_ing))
            on_hand = np.maximum(on_hand - used[k] - wasted[k], 0)
            on_hand_by_day[k] = on_hand

        snapshot_parts.append(pd.DataFrame({
            "snapshot_date": np.repeat(days.to_numpy(), n_ing),
            "store_id": s.store_id,
            "ingredient_id": np.tile(ingredient_ids, n_days),
            "quantity_on_hand": on_hand_by_day.round(2).ravel(),
            "reorder_threshold": np.tile(reorder_threshold.round(2), n_days),
        }))

    snapshots = pd.concat(snapshot_parts, ignore_index=True)
    waste = pd.DataFrame(waste_rows, columns=["waste_date", "store_id", "ingredient_id",
                                              "quantity_wasted", "waste_reason"])
    waste = waste.sort_values(["waste_date", "store_id", "ingredient_id"]).reset_index(drop=True)
    waste.insert(0, "waste_id", 1 + np.arange(len(waste)))
    return snapshots, waste


daily_usage = calculate_daily_usage(orders, order_items, recipes)
inventory_snapshots, inventory_waste_log = generate_inventory_and_waste(daily_usage, ingredients, stores, rng)
print("[inventory_snapshots] rows:", len(inventory_snapshots), "| [inventory_waste_log] rows:", len(inventory_waste_log))

# =============================================================================
# REQUIREMENTS CHECK  (matches the table in the assignment)
# =============================================================================
def show(name, rows, minimum):
    status = "OK" if rows >= minimum else "TOO FEW"
    print(f"{name:22s} {rows:>9,d}   (need {minimum:,d}+)   {status}")


print("\n================ REQUIREMENTS CHECK ================")
show("stores", len(stores), 12)
show("employees", len(employees), 60)
show("products", len(products), 40)
show("ingredients", len(ingredients), 25)
show("recipes", len(recipes), 1)
show("loyalty_members", len(loyalty_members), 2000)
show("pos_orders", len(orders), 50000)
show("pos_order_items", len(order_items), 120000)
show("shifts", len(shifts), 1)
show("inventory_snapshots", len(inventory_snapshots), 1)
show("inventory_waste_log", len(inventory_waste_log), 5000)
show("loyalty_transactions", len(loyalty_transactions), 1)

print("\nDetail checks:")
print("- cities:", stores.city.nunique(), "(need 3) | regions:", stores.region.nunique(), "(need 2+)")
print("- employees terminated:", employees.termination_date.notna().sum(),
      "| hired during the period:", (employees.hire_date >= START_DATE).sum())
print("- product categories:", products.category.nunique(), "(need 6) | ingredient types:", ingredients.ingredient_type.nunique(), "(need 5)")
sizes_per_product = recipes.groupby("product_id")["size"].apply(lambda s: s.fillna("none").nunique())
is_drink_product = products.set_index("product_id").category.isin(DRINK_CATEGORIES)
no_recipe = products[(products.category != "merch") & (~products.product_id.isin(recipes.product_id))]
print("- products (not merch) with no recipe:", len(no_recipe), "(need 0)")
print("- drinks that do not have 3 size recipes:", int(((sizes_per_product != 3) & is_drink_product.reindex(sizes_per_product.index)).sum()), "(need 0)")
months = orders.order_timestamp.dt.to_period("M").nunique()
print("- months of orders:", months, "(need 12)")
print("- average items per order:", round(len(order_items) / len(orders), 2), "(need 2+)")
store_days = orders.groupby(["store_id", "order_date"]).size().index
shift_days = set(zip(shifts.store_id, shifts.shift_start.dt.normalize()))
print("- store-days with orders but no shift:", sum(1 for k in store_days if k not in shift_days), "(need 0)")
open_days = sum(len(pd.date_range(max(START_DATE, pd.Timestamp(o)), END_DATE)) for o in stores.open_date)
print("- inventory rows:", len(inventory_snapshots), "= open store-days x ingredients =", open_days * len(ingredients))
print("- loyalty orders:", int(orders.customer_id.notna().sum()), "| earn transactions:",
      int((loyalty_transactions.txn_type == "earn").sum()), "(must match)")
t = loyalty_transactions.sort_values(["customer_id", "txn_date", "loyalty_txn_id"])
t["balance"] = t.groupby("customer_id").points.cumsum()
print("- members with negative points balance:", t[t.balance < 0].customer_id.nunique(), "(need 0)")
print("- snapshots below reorder threshold:", int((inventory_snapshots.quantity_on_hand < inventory_snapshots.reorder_threshold).sum()))


# =============================================================================
# REQUIREMENT 3 CHECKS: are the business patterns really in the data?
# =============================================================================
def run_pattern_checks():
    def verdict(ok):
        return "PASS" if ok else "REVIEW"


    print("\n================ REQUIREMENT 3: BUSINESS PATTERNS ================")
    o = orders.copy()
    o["month"] = o.order_date.dt.month
    o["weekend"] = o.order_date.dt.dayofweek >= 5
    o["drive_thru"] = o.store_id.map(dict(zip(stores.store_id, stores.has_drive_thru)))
    line = order_items.merge(products[["product_id", "product_name", "category"]], on="product_id")
    line = line.merge(product_attrs[["product_id", "temperature"]], on="product_id")
    line = line.merge(o[["order_id", "month", "store_id", "order_date"]], on="order_id")

    # 1) time of day
    by_hour = o.groupby("hour").size()
    morning = by_hour.loc[7:9].mean()
    afternoon = by_hour.loc[13:15].mean()
    evening = by_hour.loc[17:19].mean()
    print("\n1. TIME OF DAY  (avg orders per hour)  morning 7-9: %d | afternoon 13-15: %d | evening 17-19: %d"
          % (morning, afternoon, evening))
    print("   morning rush > afternoon bump > evening:", verdict(morning > afternoon > evening))

    # 2) day of week, drive-thru vs walk-in
    per_day = o.groupby(["drive_thru", "weekend", "store_id", "order_date"]).size() \
               .groupby(["drive_thru", "weekend"]).mean().round(1)
    print("\n2. DAY OF WEEK  (avg orders per store-day)\n", per_day.to_string())
    dt_ratio = per_day[(True, False)] / per_day[(True, True)]
    walk_ratio = per_day[(False, False)] / per_day[(False, True)]
    print("   weekday/weekend ratio: drive-thru %.2f vs walk-in %.2f ->" % (dt_ratio, walk_ratio),
          verdict(dt_ratio > 1 > walk_ratio))

    # 3) seasonality
    drinks = line[line.category.isin(DRINK_CATEGORIES)].copy()
    drinks["iced"] = drinks.temperature == "iced"
    iced_by_month = drinks.groupby("month").iced.mean().round(2)
    print("\n3. SEASONALITY  iced share of drinks by month:\n  ", iced_by_month.to_dict())
    print("   summer (Jun-Aug) iced share higher than winter (Dec-Feb):",
          verdict(iced_by_month.loc[6:8].mean() > iced_by_month.loc[[12, 1, 2]].mean() + 0.2))
    hot_by_month = drinks.groupby("month").apply(lambda g: (g.temperature == "hot").mean()).round(2)
    print("   hot share Oct-Dec vs Jun-Aug:", round(hot_by_month.loc[10:12].mean(), 2), "vs", round(hot_by_month.loc[6:8].mean(), 2),
          verdict(hot_by_month.loc[10:12].mean() > hot_by_month.loc[6:8].mean()))
    for name in ["Pumpkin Spice Latte", "Peppermint Mocha"]:
        share = line[line.product_name == name].groupby("month").size()
        print(f"   {name}: lines per month {share.to_dict()}")

    # 4) store variation
    store_summary = o.groupby("store_id").agg(orders=("order_id", "size"))
    store_summary["pastry_share"] = line[line.category == "pastry"].groupby("store_id").size() / line.groupby("store_id").size()
    store_summary["avg_items_per_order"] = line.groupby("store_id").size() / store_summary.orders
    store_summary["loyalty_share"] = o.groupby("store_id").customer_id.apply(lambda s: s.notna().mean())
    print("\n4. STORE VARIATION\n", store_summary.round(3).to_string())
    print("   volume differs by 30%+ between busiest and slowest store:",
          verdict(store_summary.orders.max() > 1.3 * store_summary.orders.min()))
    last_store = stores.store_id.max()
    first_sale = o[o.store_id == last_store].order_date.min().date()
    print("   mid-period store", last_store, "opens", stores.open_date.iloc[-1], "first sale", first_sale,
          verdict(pd.Timestamp(first_sale) > START_DATE + pd.Timedelta(days=90)))

    # 5) loyalty behaviour
    share = o.customer_id.notna().mean()
    print("\n5. LOYALTY  share of orders linked to a member: %.3f" % share, verdict(0.35 <= share <= 0.50))
    members_tier = loyalty_members.set_index("customer_id").current_tier
    per_member = o.dropna(subset=["customer_id"]).groupby("customer_id").size()
    tier_orders = per_member.groupby(members_tier.reindex(per_member.index)).mean().round(1)
    print("   avg orders per member by current tier:", tier_orders.to_dict(),
          verdict(tier_orders["gold"] > tier_orders["silver"] > tier_orders["bronze"]))
    items_per_order = order_items.groupby("order_id").size()
    is_member = o.set_index("order_id").customer_id.notna()
    basket = items_per_order.groupby(is_member.reindex(items_per_order.index)).mean().round(2)
    print("   avg items per order  guest %.2f vs member %.2f" % (basket[False], basket[True]), verdict(basket[True] > basket[False]))
    changed = loyalty_tier_history.groupby("customer_id").size().gt(1).sum()
    print("   members who changed tier:", int(changed), verdict(changed > 100))

    # 6) price changes
    changed_prices = product_price_history.groupby("product_id").size().gt(1).sum()
    print("\n6. PRICE CHANGES  products with a price change:", int(changed_prices), verdict(changed_prices >= 3))
    lat = line.merge(order_items[["order_item_id", "order_id", "unit_price", "size"]], on=["order_id", "order_item_id"], how="left") \
        if "order_item_id" in line.columns else None
    check = order_items.merge(o[["order_id", "order_date"]], on="order_id").merge(products[["product_id", "product_name"]], on="product_id")
    medium_latte = check[(check.product_name == "Latte") & (check["size"] == "medium")]
    before = medium_latte[medium_latte.order_date < "2025-04-01"].unit_price.unique()
    after = medium_latte[medium_latte.order_date >= "2025-04-01"].unit_price.unique()
    print("   medium Latte unit_price before Apr-1:", before, "| after:", after, verdict(list(before) == [4.5] and list(after) == [4.75]))

    # 7) cost changes
    cost_changed = ingredient_cost_history.groupby("ingredient_id").size().gt(1).sum()
    print("\n7. COST CHANGES  ingredients with a cost change:", int(cost_changed), verdict(cost_changed >= 3))

    # 8) wage changes (must show up in shifts)
    rates_per_employee = shifts.groupby("employee_id").hourly_rate_at_shift.nunique()
    print("\n8. WAGE CHANGES  employees whose rate_at_shift changes:", int((rates_per_employee > 1).sum()), verdict((rates_per_employee > 1).sum() >= 5))

    # 9) product lifecycle
    seasonal = products[products.is_seasonal]
    ok_all = True
    for p in seasonal.itertuples():
        sold = line[line.product_id == p.product_id].order_date
        inside = sold.min() >= pd.Timestamp(p.launch_date) and sold.max() <= pd.Timestamp(p.discontinued_date)
        ok_all = ok_all and inside
        print(f"9. LIFECYCLE  {p.product_name:26s} {p.launch_date} -> {p.discontinued_date} | sold {sold.min().date()} .. {sold.max().date()} {verdict(inside)}")
    print("   seasonal products launched AND discontinued inside the period:", len(seasonal), verdict(len(seasonal) >= 2 and ok_all))

    # 10) promotions
    used = order_items.dropna(subset=["promo_code"]).merge(o[["order_id", "order_date"]], on="order_id")
    print("\n10. PROMOTIONS  promo codes defined:", len(promo_codes), verdict(len(promo_codes) >= 4))
    for r in promo_codes.itertuples():
        u = used[used.promo_code == r.promo_code].order_date
        if len(u):
            inside = u.min() >= pd.Timestamp(r.start_date) and u.max() <= pd.Timestamp(r.end_date)
            print(f"    {r.promo_code:15s} {r.start_date}..{r.end_date}  used {len(u):>5d} times, {u.min().date()}..{u.max().date()} {verdict(inside)}")

    # 11) labor vs sales
    daily = o.groupby(["store_id", "order_date"]).size().rename("orders")
    staff = shifts.assign(day=shifts.shift_start.dt.normalize()).groupby(["store_id", "day"]).size().rename("shifts")
    labor = pd.concat([daily, staff.rename_axis(["store_id", "order_date"])], axis=1).dropna()
    corr = labor.orders.corr(labor.shifts)
    hours = (shifts.shift_end - shifts.shift_start).dt.total_seconds() / 3600
    labor_hours = hours.groupby(shifts.store_id).sum()
    orders_by_store = o.groupby("store_id").size()
    orders_per_labor_hour = (orders_by_store / labor_hours).round(2)
    print("\n11. LABOR vs SALES  correlation of daily orders and staff on shift: %.2f" % corr, verdict(corr > 0.5))
    print("    orders per labor hour by store (spread = some stores over/under staffed):\n   ", orders_per_labor_hour.to_dict())
    print("    best vs worst store differ by 25%+:", verdict(orders_per_labor_hour.max() > 1.25 * orders_per_labor_hour.min()))

    # 12) inventory
    snap = inventory_snapshots.copy()
    snap["low"] = snap.quantity_on_hand < snap.reorder_threshold
    share_low = snap.low.mean()
    print("\n12. INVENTORY  share of snapshots below reorder threshold: %.3f" % share_low, verdict(0.005 < share_low < 0.08))
    snap["dow"] = pd.to_datetime(snap.snapshot_date).dt.dayofweek
    espresso = snap[snap.ingredient_id == ingredients.set_index("ingredient_name").at["Espresso Beans", "ingredient_id"]]
    one_store = espresso[espresso.store_id == 1].sort_values("snapshot_date")
    print("    Espresso Beans, store 1, first 8 days of on-hand:", one_store.quantity_on_hand.head(8).round(0).tolist())
    up_days = one_store.quantity_on_hand.diff().gt(0)
    print("    refills happen only on delivery days (Mon/Thu):", verdict(set(one_store.snapshot_date.dt.dayofweek[up_days]) <= {0, 3}))


# =============================================================================
# STEP 10: REQUIREMENT 4 - REFERENTIAL INTEGRITY CHECKS (on the CLEAN tables)
# =============================================================================
integrity_failures = []


def check(name, violations):
    status = "PASS" if violations == 0 else "FAIL"
    print(f"  {status} | {name:60s} | violations: {violations}")
    if violations:
        integrity_failures.append(name)


def orphans(child_values, parent_values):
    """How many non-null child values do not exist in the parent table?"""
    child = pd.Series(child_values).dropna()
    return int((~child.isin(set(parent_values))).sum())


def run_integrity_checks():
    print("\n================ REQUIREMENT 4: INTEGRITY (clean tables, before defects) ================")
    print("Foreign keys")
    check("pos_orders.store_id -> stores", orphans(orders.store_id, stores.store_id))
    check("pos_orders.employee_id -> employees", orphans(orders.employee_id, employees.employee_id))
    check("pos_orders.customer_id -> loyalty_members", orphans(orders.customer_id, loyalty_members.customer_id))
    check("pos_order_items.order_id -> pos_orders", orphans(order_items.order_id, orders.order_id))
    check("pos_order_items.product_id -> products", orphans(order_items.product_id, products.product_id))
    check("recipes.product_id -> products", orphans(recipes.product_id, products.product_id))
    check("recipes.ingredient_id -> ingredients", orphans(recipes.ingredient_id, ingredients.ingredient_id))
    check("employees.home_store_id -> stores", orphans(employees.home_store_id, stores.store_id))
    check("shifts.employee_id -> employees", orphans(shifts.employee_id, employees.employee_id))
    check("shifts.store_id -> stores", orphans(shifts.store_id, stores.store_id))
    check("loyalty_members.signup_store_id -> stores", orphans(loyalty_members.signup_store_id, stores.store_id))
    check("loyalty_tier_history.customer_id -> loyalty_members", orphans(loyalty_tier_history.customer_id, loyalty_members.customer_id))
    check("loyalty_transactions.customer_id -> loyalty_members", orphans(loyalty_transactions.customer_id, loyalty_members.customer_id))
    check("loyalty_transactions.order_id -> pos_orders", orphans(loyalty_transactions.order_id, orders.order_id))
    check("inventory_snapshots.store_id -> stores", orphans(inventory_snapshots.store_id, stores.store_id))
    check("inventory_snapshots.ingredient_id -> ingredients", orphans(inventory_snapshots.ingredient_id, ingredients.ingredient_id))
    check("inventory_waste_log.store_id -> stores", orphans(inventory_waste_log.store_id, stores.store_id))
    check("inventory_waste_log.ingredient_id -> ingredients", orphans(inventory_waste_log.ingredient_id, ingredients.ingredient_id))

    print("Order totals reconcile with line items")
    line_total = (order_items.unit_price * order_items.quantity).groupby(order_items.order_id).sum().round(2)
    line_disc = order_items.item_discount.groupby(order_items.order_id).sum().round(2)
    o = orders.set_index("order_id")
    check("subtotal = sum(quantity x unit_price)", int(((o.subtotal - line_total).abs() > 0.005).sum()))
    check("discount_total = sum(item_discount)", int(((o.discount_total - line_disc).abs() > 0.005).sum()))
    check("tip <= subtotal", int((orders.tip > orders.subtotal).sum()))
    check("orders without any items", int((~orders.order_id.isin(order_items.order_id)).sum()))

    print("Timestamps fall inside store operating periods")
    open_date = dict(zip(stores.store_id, pd.to_datetime(stores.open_date)))
    check("orders placed before the store opened", int((orders.order_date < orders.store_id.map(open_date)).sum()))
    first_hour = np.where(orders.order_date.dt.dayofweek >= 5, 7, 6)          # weekends open at 7, weekdays at 6
    check("orders outside opening hours (6/7 AM - 8 PM)", int(((orders.hour < first_hour) | (orders.hour >= 20)).sum()))
    shift_day = shifts.shift_start.dt.normalize()
    check("shifts before the store opened", int((shift_day < shifts.store_id.map(open_date)).sum()))
    emp = employees.set_index("employee_id")
    hired = shifts.employee_id.map(emp.hire_date)
    ended = shifts.employee_id.map(emp.termination_date)
    check("shifts before the employee was hired", int((shift_day < hired).sum()))
    check("shifts after the employee was terminated", int((ended.notna() & (shift_day > ended)).sum()))
    check("shift_end <= shift_start", int((shifts.shift_end <= shifts.shift_start).sum()))
    check("inventory snapshots before the store opened",
          int((inventory_snapshots.snapshot_date < inventory_snapshots.store_id.map(open_date)).sum()))
    check("waste logged before the store opened",
          int((inventory_waste_log.waste_date < inventory_waste_log.store_id.map(open_date)).sum()))
    order_day = loyalty_transactions.order_id.map(orders.set_index("order_id").order_date)
    check("loyalty transactions dated before their order", int((loyalty_transactions.txn_date < order_day).sum()))
    signup = orders.customer_id.map(loyalty_members.set_index("customer_id").signup_date)
    check("orders placed before the member signed up", int((orders.order_date < signup).sum()))
    print("\nIntegrity result:", "ALL CLEAN" if not integrity_failures else f"{len(integrity_failures)} FAILED: {integrity_failures}")


# =============================================================================
# STEP 11: BUILD THE "AS DELIVERED" TABLES (clean, exactly the columns in the assignment)
# Every table gets a hidden "_part" column = the month (YYYY-MM) used for the file folder.
# =============================================================================
DATE_FMT = "%Y-%m-%d"
TS_FMT = "%Y-%m-%d %H:%M:%S"
FIRST_MONTH = START_DATE.strftime("%Y-%m")


def month_of(date_series):
    return date_series.dt.strftime("%Y-%m")


def text_date(series, fmt=DATE_FMT):
    """Dates are delivered as TEXT. Missing dates stay empty."""
    return pd.to_datetime(series).dt.strftime(fmt)


def build_export_tables():
    t = {}
    order_month = dict(zip(orders.order_id, month_of(orders.order_timestamp)))

    t["pos_orders"] = pd.DataFrame({
        "order_id": orders.order_id, "store_id": orders.store_id, "employee_id": orders.employee_id,
        "customer_id": orders.customer_id, "order_timestamp": orders.order_timestamp.dt.strftime(TS_FMT),
        "order_channel": orders.order_channel, "subtotal": orders.subtotal, "discount_total": orders.discount_total,
        "tax": orders.tax, "tip": orders.tip, "payment_method": orders.payment_method,
        "_part": month_of(orders.order_timestamp)})
    t["pos_order_items"] = pd.DataFrame({
        "order_item_id": order_items.order_item_id, "order_id": order_items.order_id,
        "product_id": order_items.product_id, "size": order_items["size"], "quantity": order_items.quantity,
        "unit_price": order_items.unit_price, "item_discount": order_items.item_discount,
        "promo_code": order_items.promo_code, "modifiers": order_items.modifiers,
        "_part": order_items.order_id.map(order_month)})

    signed_up_in_period = loyalty_members.signup_date >= START_DATE
    t["loyalty_members"] = pd.DataFrame({
        "customer_id": loyalty_members.customer_id, "first_name": loyalty_members.first_name,
        "last_name": loyalty_members.last_name, "email": loyalty_members.email,
        "birth_date": text_date(loyalty_members.birth_date), "signup_date": text_date(loyalty_members.signup_date),
        "signup_store_id": loyalty_members.signup_store_id, "current_tier": loyalty_members.current_tier,
        "_part": np.where(signed_up_in_period, month_of(loyalty_members.signup_date), FIRST_MONTH)})
    th = loyalty_tier_history.sort_values(["customer_id", "effective_date"]).reset_index(drop=True)
    t["loyalty_tier_history"] = pd.DataFrame({
        "customer_id": th.customer_id, "tier": th.tier, "effective_date": text_date(th.effective_date),
        "end_date": text_date(th.end_date),
        "_part": np.where(th.effective_date >= START_DATE, month_of(th.effective_date), FIRST_MONTH)})
    t["loyalty_transactions"] = pd.DataFrame({
        "loyalty_txn_id": loyalty_transactions.loyalty_txn_id, "customer_id": loyalty_transactions.customer_id,
        "order_id": loyalty_transactions.order_id, "txn_type": loyalty_transactions.txn_type,
        "points": loyalty_transactions.points, "txn_date": text_date(loyalty_transactions.txn_date),
        "_part": month_of(loyalty_transactions.txn_date)})

    t["inventory_snapshots"] = pd.DataFrame({
        "snapshot_date": text_date(inventory_snapshots.snapshot_date), "store_id": inventory_snapshots.store_id,
        "ingredient_id": inventory_snapshots.ingredient_id, "quantity_on_hand": inventory_snapshots.quantity_on_hand,
        "reorder_threshold": inventory_snapshots.reorder_threshold,
        "_part": month_of(inventory_snapshots.snapshot_date)})
    t["inventory_waste_log"] = pd.DataFrame({
        "waste_id": inventory_waste_log.waste_id, "waste_date": text_date(inventory_waste_log.waste_date),
        "store_id": inventory_waste_log.store_id, "ingredient_id": inventory_waste_log.ingredient_id,
        "quantity_wasted": inventory_waste_log.quantity_wasted, "waste_reason": inventory_waste_log.waste_reason,
        "_part": month_of(inventory_waste_log.waste_date)})
    t["shifts"] = pd.DataFrame({
        "shift_id": shifts.shift_id, "employee_id": shifts.employee_id, "store_id": shifts.store_id,
        "shift_start": shifts.shift_start.dt.strftime(TS_FMT), "shift_end": shifts.shift_end.dt.strftime(TS_FMT),
        "hourly_rate_at_shift": shifts.hourly_rate_at_shift, "_part": month_of(shifts.shift_start)})
    t["employees"] = pd.DataFrame({
        "employee_id": employees.employee_id, "first_name": employees.first_name, "last_name": employees.last_name,
        "hire_date": text_date(employees.hire_date), "termination_date": text_date(employees.termination_date),
        "role": employees.role, "home_store_id": employees.home_store_id, "hourly_rate": employees.hourly_rate,
        "_part": FIRST_MONTH})

    # reference data: one initial load (month = first month of the period)
    t["stores"] = stores.assign(_part=FIRST_MONTH)
    t["products"] = products.assign(_part=FIRST_MONTH)
    t["recipes"] = recipes.assign(_part=FIRST_MONTH)
    t["ingredients"] = ingredients.assign(_part=FIRST_MONTH)

    # extra history tables (allowed: "you may add tables"): the exports above only hold CURRENT price/cost/rate
    for name, df in [("product_price_history", product_price_history),
                     ("ingredient_cost_history", ingredient_cost_history),
                     ("employee_rate_history", employee_rate_history)]:
        df = df.copy()
        df["effective_date"] = text_date(df.effective_date)
        df["end_date"] = text_date(df.end_date)
        t[name] = df.assign(_part=FIRST_MONTH)
    promo = promo_codes.copy()
    t["promo_codes"] = promo.assign(_part=FIRST_MONTH)
    return t


# =============================================================================
# STEP 12: INTENTIONAL DEFECTS (Requirement 5)
# Each defect is a small function. Rates come from config.yaml (defects.rates.<name>).
# =============================================================================
defect_log = []


def get_rate(name):
    return float(DEFECT_CONFIG.get("rates", {}).get(name, DEFECT_CONFIG["rate"]))


def pick_rows(n_rows, rate):
    """Randomly choose which rows (by position) get the defect."""
    k = int(round(n_rows * rate))
    return rng.choice(n_rows, size=k, replace=False) if k > 0 else np.array([], dtype=int)


def log_defect(number, kind, table, column, rate, injected, eligible, note=""):
    defect_log.append({"defect_no": number, "defect_type": kind, "table": table, "column": column,
                       "configured_rate": rate, "rows_eligible": eligible, "rows_injected": int(injected),
                       "actual_rate": round(injected / eligible, 5) if eligible else 0, "note": note})


def set_cell(df, row, column, value):
    df.iat[row, df.columns.get_loc(column)] = value


def as_text_column(df, column):
    df[column] = df[column].astype(object)          # lets the column hold both numbers and text


# ---- 9. logically impossible values
def defect_shift_end_before_start(df):
    rate = get_rate("impossible_shift_end")
    rows = pick_rows(len(df), rate)
    for i in rows:
        start = pd.Timestamp(df["shift_start"].iat[i])
        set_cell(df, i, "shift_end", (start - pd.Timedelta(hours=int(rng.integers(1, 5)))).strftime(TS_FMT))
    log_defect(9, "Logically impossible", "shifts", "shift_end", rate, len(rows), len(df), "shift_end earlier than shift_start")


def defect_tip_above_subtotal(df):
    rate = get_rate("impossible_tip")
    as_text_column(df, "tip")
    rows = pick_rows(len(df), rate)
    for i in rows:
        set_cell(df, i, "tip", round(float(df["subtotal"].iat[i]) * 3 + 5, 2))
    log_defect(9, "Logically impossible", "pos_orders", "tip", rate, len(rows), len(df), "tip greater than subtotal")


# ---- 12. outliers
def defect_outliers(df):
    as_text_column(df, "quantity")
    as_text_column(df, "unit_price")
    for column, value, name in [("quantity", 500, "outlier_quantity"), ("unit_price", 9999, "outlier_unit_price")]:
        rate = get_rate(name)
        rows = pick_rows(len(df), rate)
        for i in rows:
            set_cell(df, i, column, value)
        log_defect(12, "Outlier", "pos_order_items", column, rate, len(rows), len(df), f"{column} set to {value}")


# ---- 7. negative or zero quantities
def defect_bad_quantities(df):
    as_text_column(df, "quantity")
    for value, name, note in [(-1, "negative_quantity", "void/refund stored as -1"), (0, "zero_quantity", "quantity = 0")]:
        rate = get_rate(name)
        rows = pick_rows(len(df), rate)
        for i in rows:
            set_cell(df, i, "quantity", value)
        log_defect(7, "Negative or zero quantity", "pos_order_items", "quantity", rate, len(rows), len(df), note)


def defect_negative_waste(df):
    rate = get_rate("negative_waste_quantity")
    as_text_column(df, "quantity_wasted")
    rows = pick_rows(len(df), rate)
    for i in rows:
        set_cell(df, i, "quantity_wasted", -abs(float(df["quantity_wasted"].iat[i])))
    log_defect(7, "Negative or zero quantity", "inventory_waste_log", "quantity_wasted", rate, len(rows), len(df), "waste quantity made negative")


# ---- 8. late-arriving records
def defect_late_loyalty_txn(df):
    rate = get_rate("late_loyalty_txn")
    as_text_column(df, "txn_date")
    eligible = np.where(df["txn_type"].isin(["earn", "redeem"]).to_numpy())[0]
    rows = rng.choice(eligible, size=int(round(len(eligible) * rate)), replace=False)
    for i in rows:
        earlier = pd.Timestamp(df["txn_date"].iat[i]) - pd.Timedelta(days=int(rng.integers(3, 11)))
        set_cell(df, i, "txn_date", earlier.strftime(DATE_FMT))
    log_defect(8, "Late-arriving record", "loyalty_transactions", "txn_date", rate, len(rows), len(eligible),
               "txn_date 3-10 days BEFORE its order; file stays in the order's month")


# ---- 10. overlapping history
def defect_overlapping_tiers(df):
    rate = get_rate("overlapping_tier_history")
    as_text_column(df, "end_date")
    same_customer_next = (df["customer_id"].to_numpy()[:-1] == df["customer_id"].to_numpy()[1:])
    has_end = df["end_date"].notna().to_numpy()[:-1]
    eligible = np.where(same_customer_next & has_end)[0]
    rows = rng.choice(eligible, size=int(round(len(eligible) * rate)), replace=False)
    for i in rows:
        next_start = pd.Timestamp(df["effective_date"].iat[i + 1])
        set_cell(df, i, "end_date", (next_start + pd.Timedelta(days=int(rng.integers(5, 30)))).strftime(DATE_FMT))
    log_defect(10, "Overlapping history", "loyalty_tier_history", "end_date", rate, len(rows), len(eligible),
               "a tier's end_date runs past the next tier's effective_date")


# ---- 6. unexpected nulls
def defect_nulls(df, table, column, rate_name, only_where=None):
    rate = get_rate(rate_name)
    as_text_column(df, column)
    pool = np.where(only_where.to_numpy())[0] if only_where is not None else np.arange(len(df))
    rows = rng.choice(pool, size=int(round(len(pool) * rate)), replace=False)
    for i in rows:
        set_cell(df, i, column, None)
    log_defect(6, "Unexpected null", table, column, rate, len(rows), len(pool))


# ---- 5. orphaned foreign keys
def defect_orphans(df, table, column, rate_name, fake_values):
    rate = get_rate(rate_name)
    as_text_column(df, column)
    rows = pick_rows(len(df), rate)
    for i in rows:
        set_cell(df, i, column, int(rng.choice(fake_values)))
    log_defect(5, "Orphaned foreign key", table, column, rate, len(rows), len(df), f"values like {fake_values[0]} do not exist in the parent table")


# ---- 4. numbers stored as strings
def defect_money_strings(df, table, column, rate_name):
    rate = get_rate(rate_name)
    as_text_column(df, column)
    rows = pick_rows(len(df), rate)
    for i in rows:
        value = df[column].iat[i]
        if value is None or pd.isna(value):
            continue
        value = float(value)
        styles = [f"${value:.2f}", f"{value:.2f}".replace(".", ","), f"{value:.2f} USD"]
        set_cell(df, i, column, styles[int(rng.integers(0, 3))])
    log_defect(4, "Number stored as string", table, column, rate, len(rows), len(df), 'examples: "$4.50", "4,50", "4.50 USD"')


# ---- 3. mixed date/time formats
def defect_mixed_dates(df, table, column, rate_name, has_time):
    rate = get_rate(rate_name)
    as_text_column(df, column)
    formats = ["%m/%d/%Y %I:%M %p", "%d-%b-%y %H:%M", "%Y/%m/%d %H:%M:%S"] if has_time else ["%m/%d/%Y", "%d-%b-%y", "%Y.%m.%d"]
    rows = pick_rows(len(df), rate)
    for i in rows:
        set_cell(df, i, column, pd.Timestamp(df[column].iat[i]).strftime(formats[int(rng.integers(0, 3))]))
    log_defect(3, "Mixed date/time format", table, column, rate, len(rows), len(df), "examples: " + ", ".join(formats))


# ---- 2. inconsistent casing / whitespace
def defect_casing(df, table, column, rate_name):
    rate = get_rate(rate_name)
    as_text_column(df, column)
    rows = pick_rows(len(df), rate)
    count = 0
    for i in rows:
        value = df[column].iat[i]
        if not isinstance(value, str):
            continue
        messy = [value.upper(), value.title(), value + " ", " " + value, value.upper() + " "]
        set_cell(df, i, column, messy[int(rng.integers(0, 5))])
        count += 1
    log_defect(2, "Inconsistent casing/whitespace", table, column, rate, count, len(df), "e.g. credit, Credit, CREDIT , ' credit'")


# ---- 1. duplicate records (LAST: an exact replay of rows that already contain other defects)
def defect_duplicates(df, table, rate_name):
    rate = get_rate(rate_name)
    rows = pick_rows(len(df), rate)
    result = pd.concat([df, df.iloc[rows]], ignore_index=True)
    log_defect(1, "Duplicate record", table, "(entire row)", rate, len(rows), len(df), "exact duplicate appended (retry/replay)")
    return result


def inject_defects(tables):
    """Takes the clean tables, returns messy copies. Order matters: change values first, change formats later."""
    d = {name: df.copy() for name, df in tables.items()}
    if not DEFECT_CONFIG["enabled"]:
        return d
    defect_shift_end_before_start(d["shifts"])
    defect_tip_above_subtotal(d["pos_orders"])
    defect_outliers(d["pos_order_items"])
    defect_bad_quantities(d["pos_order_items"])
    defect_negative_waste(d["inventory_waste_log"])
    defect_late_loyalty_txn(d["loyalty_transactions"])
    defect_overlapping_tiers(d["loyalty_tier_history"])
    defect_nulls(d["inventory_waste_log"], "inventory_waste_log", "waste_reason", "null_waste_reason")
    drinks = d["pos_order_items"]["size"].notna()
    defect_nulls(d["pos_order_items"], "pos_order_items", "size", "null_drink_size", only_where=drinks)
    defect_nulls(d["pos_orders"], "pos_orders", "payment_method", "null_payment_method")
    defect_nulls(d["loyalty_members"], "loyalty_members", "email", "null_email")
    defect_orphans(d["pos_order_items"], "pos_order_items", "product_id", "orphan_item_product", [900, 901, 999, 5000])
    defect_orphans(d["pos_orders"], "pos_orders", "employee_id", "orphan_order_employee", [9999, 9998])
    defect_orphans(d["inventory_waste_log"], "inventory_waste_log", "ingredient_id", "orphan_waste_ingredient", [777, 888])
    defect_money_strings(d["pos_orders"], "pos_orders", "subtotal", "string_subtotal")
    defect_money_strings(d["pos_orders"], "pos_orders", "tip", "string_tip")
    defect_money_strings(d["pos_order_items"], "pos_order_items", "unit_price", "string_unit_price")
    defect_money_strings(d["employees"], "employees", "hourly_rate", "string_hourly_rate")
    defect_mixed_dates(d["pos_orders"], "pos_orders", "order_timestamp", "mixed_order_timestamp", True)
    defect_mixed_dates(d["loyalty_members"], "loyalty_members", "signup_date", "mixed_signup_date", False)
    defect_mixed_dates(d["shifts"], "shifts", "shift_start", "mixed_shift_start", True)
    defect_mixed_dates(d["inventory_waste_log"], "inventory_waste_log", "waste_date", "mixed_waste_date", False)
    defect_casing(d["pos_orders"], "pos_orders", "payment_method", "casing_payment_method")
    defect_casing(d["pos_orders"], "pos_orders", "order_channel", "casing_order_channel")
    defect_casing(d["products"], "products", "category", "casing_product_category")
    defect_casing(d["employees"], "employees", "role", "casing_employee_role")
    defect_casing(d["inventory_waste_log"], "inventory_waste_log", "waste_reason", "casing_waste_reason")
    defect_casing(d["loyalty_members"], "loyalty_members", "current_tier", "casing_current_tier")
    d["pos_orders"] = defect_duplicates(d["pos_orders"], "pos_orders", "duplicate_orders")
    d["pos_order_items"] = defect_duplicates(d["pos_order_items"], "pos_order_items", "duplicate_items")
    d["loyalty_transactions"] = defect_duplicates(d["loyalty_transactions"], "loyalty_transactions", "duplicate_loyalty_txns")
    return d


# =============================================================================
# STEP 13: WRITE THE BRONZE FILES (Requirement 6)
#   bronze/<source>/<table>/ingest_month=YYYY-MM/<table>_<YYYYMMDD>.<ext>
# =============================================================================
import json
import shutil

# table -> (source folder, _source_system value, file format)
FILE_LAYOUT = {
    "pos_orders": ("pos", "pos", "csv"),
    "pos_order_items": ("pos", "pos", "csv"),
    "loyalty_members": ("loyalty", "loyalty_app", "jsonl"),
    "loyalty_tier_history": ("loyalty", "loyalty_app", "jsonl"),
    "loyalty_transactions": ("loyalty", "loyalty_app", "jsonl"),
    "ingredients": ("inventory", "inventory", "csv"),
    "inventory_snapshots": ("inventory", "inventory", "parquet"),
    "inventory_waste_log": ("inventory", "inventory", "csv"),
    "employees": ("hr", "hr", "csv"),
    "shifts": ("hr", "hr", "parquet"),
    "employee_rate_history": ("hr", "hr", "csv"),
    "stores": ("reference", "product_catalog", "csv"),
    "products": ("reference", "product_catalog", "csv"),
    "recipes": ("reference", "product_catalog", "csv"),
    "product_price_history": ("reference", "product_catalog", "csv"),
    "ingredient_cost_history": ("reference", "inventory", "csv"),
    "promo_codes": ("reference", "product_catalog", "csv"),
}
SCHEMA_DRIFT_FROM = "2025-07"      # from this month on, files have a renamed / extra column


def get_output_dir():
    out = Path(DATA_CONFIG.get("output_dir", "bronze"))
    return out if out.is_absolute() else BASE_DIR / out


def prepare_output_dir(out):
    if out.exists():
        # safety: only delete a folder that is empty or was created by this generator
        if any(out.iterdir()) and not (out / "_row_counts.csv").exists():
            raise SystemExit(f"Refusing to delete '{out}': it is not empty and is not a previous bronze folder.")
        shutil.rmtree(out)
    out.mkdir(parents=True)


def write_one_file(g, path, file_format):
    if file_format == "csv":
        g.to_csv(path, index=False)
    elif file_format == "jsonl":
        g = g.astype(object).where(g.notna(), None)                 # missing values become JSON null
        with open(path, "w") as f:
            for record in g.to_dict(orient="records"):
                f.write(json.dumps(record, default=str) + "\n")
    else:                                                           # parquet
        for column in g.columns:
            if g[column].dtype == object:
                g[column] = g[column].astype("string")              # mixed text/numbers -> text
        g.to_parquet(path, index=False)


def write_bronze(tables, out):
    row_counts = []
    for table, df in tables.items():
        folder, system, file_format = FILE_LAYOUT[table]
        for month, g in df.groupby("_part", sort=True):
            g = g.drop(columns="_part").copy()
            month_end = (pd.Timestamp(month + "-01") + pd.offsets.MonthEnd(0)).strftime("%Y%m%d")
            file_name = f"{table}_{month_end}.{file_format}"

            # defect 11 (schema drift): column renamed / added partway through the period
            if table == "pos_orders" and month >= SCHEMA_DRIFT_FROM:
                g = g.rename(columns={"order_channel": "channel"})
            if table == "loyalty_members" and month >= SCHEMA_DRIFT_FROM:
                g["marketing_opt_in"] = "Y"

            # ingestion metadata (Requirement 6)
            g["_source_system"] = system
            g["_source_file"] = file_name
            g["_batch_id"] = f"{system}_{month.replace('-', '')}_seed{SEED}"
            g["_ingested_at"] = f"{month_end[:4]}-{month_end[4:6]}-{month_end[6:]} 23:59:00"

            folder_path = out / folder / table / f"ingest_month={month}"
            folder_path.mkdir(parents=True, exist_ok=True)
            write_one_file(g, folder_path / file_name, file_format)
        row_counts.append({"source": folder, "table": table, "format": file_format,
                           "files": df["_part"].nunique(), "rows": len(df)})
    return pd.DataFrame(row_counts)


def write_sample_rows(tables, out):
    """First 10 rows of every table (a deliverable)."""
    with open(out / "_sample_first10_rows.txt", "w") as f:
        for table, df in tables.items():
            f.write(f"\n===== {table} ({len(df):,} rows) =====\n")
            f.write(df.drop(columns="_part").head(10).to_string(index=False) + "\n")


# =============================================================================
# RUN EVERYTHING
# =============================================================================
if DATA_CONFIG.get("show_checks", True):
    run_pattern_checks()
run_integrity_checks()

clean_tables = build_export_tables()
bronze_tables = inject_defects(clean_tables)

# defect 11 (schema drift) is applied while writing, so log it here
log_defect(11, "Schema drift", "pos_orders", "order_channel -> channel", "from 2025-07",
           int((bronze_tables["pos_orders"]["_part"] >= SCHEMA_DRIFT_FROM).sum()), len(bronze_tables["pos_orders"]),
           f"column renamed in every file from ingest_month={SCHEMA_DRIFT_FROM}")
log_defect(11, "Schema drift", "loyalty_members", "marketing_opt_in (new column)", "from 2025-07",
           int((bronze_tables["loyalty_members"]["_part"] >= SCHEMA_DRIFT_FROM).sum()), len(bronze_tables["loyalty_members"]),
           f"extra column in every file from ingest_month={SCHEMA_DRIFT_FROM}")

OUTPUT_DIR = get_output_dir()
prepare_output_dir(OUTPUT_DIR)
row_counts = write_bronze(bronze_tables, OUTPUT_DIR)
row_counts.to_csv(OUTPUT_DIR / "_row_counts.csv", index=False)
defect_table = pd.DataFrame(defect_log).sort_values(["defect_no", "table", "column"])
defect_table.to_csv(OUTPUT_DIR / "_defect_log.csv", index=False)
write_sample_rows(bronze_tables, OUTPUT_DIR)

print("\n================ BRONZE FILES WRITTEN ================")
print("Folder:", OUTPUT_DIR)
print(row_counts.to_string(index=False))
print("\nDefect log (type, table, column, configured rate, rows injected):")
print(defect_table[["defect_no", "defect_type", "table", "column", "configured_rate", "rows_injected"]].to_string(index=False))
print("\nDefect types injected:", defect_table.defect_no.nunique(), "(need 8+)")
print("\nDone. Same seed + same config = identical files.")